In [1]:
import os
HOME = os.getcwd()
print(HOME)
%pip install ultralytics

c:\Users\rizro\HulaDroneControlApp-desktop
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "ultralytics"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)


Name: ultralytics
Version: 8.4.3
Summary: Ultralytics YOLO ðŸš€ for SOTA object detection, multi-object tracking, instance segmentation, pose estimation and image classification.
Home-page: 
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: c:\Users\rizro\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: matplotlib, numpy, opencv-python, pillow, polars, psutil, pyyaml, requests, scipy, torch, torch, torchvision, ultralytics-thop
Required-by: 




In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
'''import subprocess, sys

result = subprocess.run([
    sys.executable, "-m", "pip", "install",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--force-reinstall"
], capture_output=True, text=True)

print("STDOUT:", result.stdout[-3000:])
print("STDERR:", result.stderr[-3000:])
print("Return code:", result.returncode)'''

'import subprocess, sys\n\nresult = subprocess.run([\n    sys.executable, "-m", "pip", "install",\n    "torch", "torchvision", "torchaudio",\n    "--index-url", "https://download.pytorch.org/whl/cu124",\n    "--force-reinstall"\n], capture_output=True, text=True)\n\nprint("STDOUT:", result.stdout[-3000:])\nprint("STDERR:", result.stderr[-3000:])\nprint("Return code:", result.returncode)'

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s-obb.pt')

results = model.train(
    data=r"C:\Users\rizro\Downloads\aeromaze_obb.v5i.yolov8-obb\data.yaml",

    # --- Duration ---
    epochs=300,
    patience=35,

    # --- Input ---
    imgsz=640,
    batch=4,
    cache=True,
    workers=0,         # Windows fix: prevents MemoryError at close_mosaic dataloader reset

    # --- Device ---
    device=0,

    # --- Learning rate (matched to pose) ---
    optimizer='Adam',
    cos_lr=True,
    lr0=0.0005,
    lrf=0.01,

    # --- Regularisation (matched to pose) ---
    dropout=0.1,
    weight_decay=0.001,
    cls=1.0,           # matched to pose — stronger classification signal for mirror pairs (3/4, 8/9)

    # --- Augmentation ---
    hsv_h=0.015,
    hsv_s=0.0,         # grayscale dataset — saturation shift has zero effect
    hsv_v=0.3,         # matched to pose
    fliplr=0.0,        # CRITICAL: signs 3/4 and 8/9 are horizontal mirror pairs
    flipud=0.0,        
    degrees=90,        # covers sideways signs (±90°); OBB angle head handles this range well
    scale=0.5,         # matched to pose
    mosaic=0.5,
    close_mosaic=20,
    # angle left at default 1.0 — train18 (best ever run) used this value
)

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.3  Python-3.12.1 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=20, cls=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\rizro\Downloads\aeromaze_obb.v5i.yolov8-obb\data.yaml, degrees=90, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.0, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-obb.pt, momentum=0.937, mosaic=

In [6]:
from ultralytics import YOLO

best_weights = str(results.save_dir / 'weights' / 'best.pt')
print(f"Evaluating: {best_weights}")

eval_model = YOLO(best_weights)
metrics = eval_model.val(data=r"C:\Users\rizro\Downloads\aeromaze_obb.v5i.yolov8-obb\data.yaml", device=0)

print(f"\n=== Best Model — Validation Metrics ===")
print(f"  mAP50:     {metrics.box.map50:.4f}")
print(f"  mAP50-95:  {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall:    {metrics.box.mr:.4f}")

names = eval_model.names
print(f"\n  Per-class mAP50:")
for i, ap in enumerate(metrics.box.ap50):
    print(f"    class {names[i]:>3}: {ap:.4f}")

Evaluating: C:\Users\rizro\HulaDroneControlApp-desktop\runs\obb\train36\weights\best.pt
Ultralytics 8.4.3  Python-3.12.1 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
YOLOv8s-obb summary (fused): 82 layers, 11,415,441 parameters, 0 gradients, 29.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 123.962.1 MB/s, size: 110.0 KB)
val: Scanning C:\Users\rizro\Downloads\aeromaze_obb.v5i.yolov8-obb\valid\labels.cache... 150 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.0s.9s
                   all        150        709      0.984      0.983      0.992      0.953
                     1         67         67      0.995      0.985      0.993      0.946
                    10         63         63      0.953      0.984      0.988      0.952
                     2         67         67      0.997          1      0.995     